# Gold Layer — Patient & Care Quality KPI Aggregations
Department summary, readmission analysis, patient flow, staff utilisation, weekly trends, outcome scorecards.

In [ ]:
spark.conf.set('spark.sql.parquet.vorder.default', 'true')
spark.conf.set('spark.databricks.delta.optimizeWrite.enabled', 'true')

In [ ]:
from pyspark.sql.functions import (
    col, sum as _sum, avg as _avg, count, countDistinct,
    min as _min, max as _max, round as _round, when, lit,
    current_timestamp, weekofyear, month, dense_rank, lag
)
from pyspark.sql.window import Window

In [ ]:
# === TABLE 1: Daily Department Summary ===
df_adm = spark.read.format('delta').table('silver_patient_admissions')

gold_dept = (
    df_adm
    .groupBy('admission_date_only', 'department', 'admission_type')
    .agg(
        count('*').alias('admissions'),
        _sum(col('is_readmission').cast('int')).alias('readmissions'),
        _avg('length_of_stay_days').alias('avg_los'),
        _max('length_of_stay_days').alias('max_los'),
        _sum(col('high_risk_flag').cast('int')).alias('high_risk_count'),
        countDistinct('primary_dx_code').alias('unique_diagnoses'),
    )
    .withColumn('readmission_rate', _round(col('readmissions') / col('admissions') * 100, 2))
    .withColumn('avg_los', _round(col('avg_los'), 2))
    .withColumn('admission_week', weekofyear(col('admission_date_only').cast('date')))
    .withColumn('admission_month', month(col('admission_date_only').cast('date')))
    .withColumn('dept_performance',
        when(col('readmission_rate') <= 5, 'Excellent')
        .when(col('readmission_rate') <= 10, 'Good')
        .when(col('readmission_rate') <= 15, 'Needs Improvement')
        .otherwise('Critical'))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_dept.write.mode('overwrite').format('delta').saveAsTable('gold_department_daily_summary')
print(f'Gold department daily summary: {gold_dept.count()} rows')

In [ ]:
# === TABLE 2: Readmission Analysis by Diagnosis & Insurance ===
gold_readmit = (
    df_adm
    .groupBy('primary_dx_code', 'insurance_type', 'age_group')
    .agg(
        count('*').alias('total_admissions'),
        _sum(col('is_readmission').cast('int')).alias('readmissions'),
        _avg('length_of_stay_days').alias('avg_los'),
        _sum(col('high_risk_flag').cast('int')).alias('high_risk_count'),
    )
    .withColumn('readmission_rate', _round(col('readmissions') / col('total_admissions') * 100, 2))
    .withColumn('avg_los', _round(col('avg_los'), 2))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_readmit.write.mode('overwrite').format('delta').saveAsTable('gold_readmission_analysis')
print(f'Gold readmission analysis: {gold_readmit.count()} rows')

In [ ]:
# === TABLE 3: Patient Flow by Shift & Department ===
gold_flow = (
    df_adm
    .groupBy('admission_shift', 'department', 'los_bucket')
    .agg(
        count('*').alias('patient_count'),
        _avg('length_of_stay_days').alias('avg_los'),
        _sum(col('is_readmission').cast('int')).alias('readmissions'),
    )
    .withColumn('avg_los', _round(col('avg_los'), 2))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_flow.write.mode('overwrite').format('delta').saveAsTable('gold_patient_flow')
print(f'Gold patient flow: {gold_flow.count()} rows')

In [ ]:
# === TABLE 4: Staff Utilisation ===
gold_staff = (
    df_adm
    .groupBy('assigned_staff_id', 'department')
    .agg(
        count('*').alias('patients_handled'),
        _avg('length_of_stay_days').alias('avg_patient_los'),
        _sum(col('is_readmission').cast('int')).alias('readmissions_handled'),
        _sum(col('high_risk_flag').cast('int')).alias('high_risk_patients'),
    )
    .withColumn('avg_patient_los', _round(col('avg_patient_los'), 2))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_staff.write.mode('overwrite').format('delta').saveAsTable('gold_staff_utilisation')
print(f'Gold staff utilisation: {gold_staff.count()} rows')

In [ ]:
# === TABLE 5: Weekly Trends ===
weekly = (
    df_adm
    .groupBy('admission_week', 'department')
    .agg(
        count('*').alias('weekly_admissions'),
        _sum(col('is_readmission').cast('int')).alias('weekly_readmissions'),
        _avg('length_of_stay_days').alias('avg_los'),
    )
    .withColumn('readmission_rate', _round(col('weekly_readmissions') / col('weekly_admissions') * 100, 2))
    .withColumn('avg_los', _round(col('avg_los'), 2))
)

w_trend = Window.partitionBy('department').orderBy('admission_week')
gold_weekly = (
    weekly
    .withColumn('prev_week_admissions', lag('weekly_admissions', 1).over(w_trend))
    .withColumn('wow_change_pct',
        _round((col('weekly_admissions') - col('prev_week_admissions')) / col('prev_week_admissions') * 100, 2))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_weekly.write.mode('overwrite').format('delta').saveAsTable('gold_weekly_trends')
print(f'Gold weekly trends: {gold_weekly.count()} rows')

In [ ]:
# === TABLE 6: Department Scorecard (overall) ===
w_rank = Window.orderBy(col('readmission_rate').asc())

gold_scorecard = (
    df_adm
    .groupBy('department')
    .agg(
        count('*').alias('total_admissions'),
        _sum(col('is_readmission').cast('int')).alias('total_readmissions'),
        _avg('length_of_stay_days').alias('avg_los'),
        _sum(col('high_risk_flag').cast('int')).alias('high_risk_count'),
        countDistinct('primary_dx_code').alias('unique_diagnoses'),
        countDistinct('admission_date_only').alias('active_days'),
    )
    .withColumn('readmission_rate', _round(col('total_readmissions') / col('total_admissions') * 100, 2))
    .withColumn('avg_los', _round(col('avg_los'), 2))
    .withColumn('daily_admissions_avg', _round(col('total_admissions') / col('active_days'), 1))
    .withColumn('performance_rank', dense_rank().over(w_rank))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_scorecard.write.mode('overwrite').format('delta').saveAsTable('gold_department_scorecard')
print(f'Gold department scorecard: {gold_scorecard.count()} rows')